In [ ]:
# Parameters (will be overridden by URL query params when running with Voila)
import os

# Default fallback End-to-End ID to ensure data is returned
endToEndId = os.getenv("ALERT_HISTORY_E2E_ID", "05c7ead85a1343d5a959561523a965fb")
backendUrl = os.getenv("CASE_MGMT_BACKEND_URL", "http://localhost:3000")
tenantId = os.getenv("ALERT_HISTORY_TENANT_ID", "DEFAULT")
dateRange = "all"       # 30days, 90days, 6months, 1year, all
granularity = "day"     # day, week, month, year
page = 1
limit = 200

In [ ]:
try:
    import requests
    import pandas as pd
    import plotly.graph_objects as go
    import plotly.express as px
    import plotly.io as pio
    from plotly.subplots import make_subplots
    from IPython.display import display, HTML
    
    # Set default renderer to basic html/notebook
    pio.renderers.default = "notebook"

    # Output styling
    display(HTML("""
    <style>
        body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
        .metric-card { background: white; padding: 15px; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); text-align: center; }
        .metric-value { font-size: 24px; font-weight: bold; color: #111827; }
        .metric-label { font-size: 12px; color: #6B7280; text-transform: uppercase; letter-spacing: 0.05em; }
        .grid-container { display: grid; grid-template-columns: repeat(5, 1fr); gap: 15px; margin-bottom: 20px; }
        .table-container { margin-top: 20px; overflow-x: auto; }
        table { width: 100%; border-collapse: collapse; font-size: 14px; }
        th { text-align: left; padding: 12px; border-bottom: 1px solid #e5e7eb; color: #6B7280; font-weight: 600; }
        td { padding: 12px; border-bottom: 1px solid #e5e7eb; color: #111827; }
        tr:last-child td { border-bottom: none; }
    </style>
    """))
except Exception as e:
    print(f"Import Error: {e}")

In [ ]:
def fetch_json(endpoint: str, params: dict):
    url = f"{backendUrl}{endpoint}"
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    return resp.json()

summary = {}
timeline = {}
alerts_resp = {}

try:
    base_params = {
        'endToEndId': endToEndId,
        'tenantId': tenantId,
    }

    summary = fetch_json(
        "/api/v1/lakehouse/alert-history/summary",
        {**base_params, 'dateRange': dateRange},
    )

    timeline = fetch_json(
        "/api/v1/lakehouse/alert-history/timeline",
        {**base_params, 'dateRange': dateRange, 'granularity': granularity},
    )

    alerts_resp = fetch_json(
        "/api/v1/lakehouse/alert-history/alerts",
        {**base_params, 'dateRange': dateRange, 'page': page, 'limit': limit},
    )

    # DataFrames
    alert_count = timeline.get('alertCountOverTime', [])
    alert_value = timeline.get('alertValueOverTime', [])
    alerts = alerts_resp.get('alerts', [])

    df_counts = pd.DataFrame(alert_count)
    if not df_counts.empty:
        df_counts['date'] = pd.to_datetime(df_counts['date'])

    df_value = pd.DataFrame(alert_value)
    if not df_value.empty:
        df_value['date'] = pd.to_datetime(df_value['date'])

    df_alerts = pd.DataFrame(alerts)
    if not df_alerts.empty:
        df_alerts['date'] = pd.to_datetime(df_alerts['date'])

except Exception as e:
    display(HTML(f"<div style='color: red; padding: 20px; border: 1px solid red; border-radius: 5px;'>Error fetching data: {str(e)}</div>"))
    print(f"Backend URL: {backendUrl}")

In [ ]:
if summary:
    total_alerts = f"{summary.get('totalAlerts', 0):,}"
    cases_opened = f"{summary.get('casesOpened', 0):,}"
    investigations = f"{summary.get('investigations', 0):,}"
    sar_filings = f"{summary.get('sarFilings', 0):,}"
    total_value = f"${summary.get('totalValue', 0):,.2f}"

    display(HTML(f"""
    <div class="grid-container">
        <div class="metric-card">
            <div class="metric-label">Total Alerts</div>
            <div class="metric-value">{total_alerts}</div>
        </div>
        <div class="metric-card">
            <div class="metric-label">Cases Opened</div>
            <div class="metric-value">{cases_opened}</div>
        </div>
        <div class="metric-card">
            <div class="metric-label">Investigations</div>
            <div class="metric-value">{investigations}</div>
        </div>
        <div class="metric-card">
            <div class="metric-label">SAR/STR Filings</div>
            <div class="metric-value">{sar_filings}</div>
        </div>
        <div class="metric-card">
            <div class="metric-label">Total Value</div>
            <div class="metric-value">{total_value}</div>
        </div>
    </div>
    """))

In [ ]:
if (not df_counts.empty) or (not df_value.empty):
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=False,
        vertical_spacing=0.12,
        subplot_titles=(
            "Alert Count Over Time",
            "Alert Value Over Time",
        ),
        row_heights=[0.5, 0.5],
    )

    # Alert Count Over Time (multi-line time series)
    if not df_counts.empty:
        fig.add_trace(
            go.Scatter(
                x=df_counts['date'],
                y=df_counts['alerts'],
                mode='lines+markers',
                name='Alerts',
                line=dict(color='#ef4444', width=2.5),
                marker=dict(size=6),
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=df_counts['date'],
                y=df_counts['cases'],
                mode='lines+markers',
                name='Cases',
                line=dict(color='#3b82f6', width=2, dash='dot'),
                marker=dict(size=5),
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=df_counts['date'],
                y=df_counts['investigations'],
                mode='lines+markers',
                name='Investigations',
                line=dict(color='#8b5cf6', width=2, dash='dash'),
                marker=dict(size=5),
            ),
            row=1,
            col=1,
        )

    # Alert Value Over Time (bar chart, one bar per day)
    if not df_value.empty:
        fig.add_trace(
            go.Bar(
                x=df_value['date'],
                y=df_value['totalValue'],
                name='Total Alert Value',
                marker_color='#10b981',
            ),
            row=2,
            col=1,
        )

    fig.update_layout(
        height=720,
        showlegend=True,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        margin=dict(l=20, r=20, t=60, b=20),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
    )

    # Axes
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#e5e7eb', row=1, col=1)
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#e5e7eb', row=2, col=1)
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#e5e7eb', row=1, col=1, title_text="Count")
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#e5e7eb', row=2, col=1, title_text="Total Value")

    fig.show()

In [ ]:
if not df_alerts.empty:
    display(HTML("<h3>Alert History</h3>"))

    # Format columns for readability
    display_df = df_alerts.copy()
    if 'date' in display_df.columns:
        display_df['date'] = pd.to_datetime(display_df['date']).dt.strftime('%Y-%m-%d')

    rename_map = {
        'alertId': 'Alert ID',
        'type': 'Type',
        'severity': 'Severity',
        'status': 'Status',
        'caseId': 'Case ID',
        'outcome': 'Outcome',
        'date': 'Date',
    }

    display_df = display_df[[c for c in rename_map.keys() if c in display_df.columns]].rename(columns=rename_map)

    # Render HTML table
    html = display_df.to_html(index=False, classes='table')
    display(HTML(f"<div class='table-container'>{html}</div>"))
else:
    display(HTML("<p>No alerts found for this Transaction.</p>"))